# 📘 Notebook 04 — Model Evaluation & Cost Asymmetry Analysis

## 🏢 STAGE 16 — Comprehensive Evaluation & Financial Risk
In this notebook, we examine model performance across **7 Diagnostic Dimensions** and connect statistical classification metrics directly to **Business Cost & Financial Asymmetry**.

> **The Core Evaluation Insight:** *In customer churn, classification errors carry highly asymmetric costs: losing a customer (False Negative $\approx \$1,800$ loss) is 36 times more expensive than sending an unnecessary discount voucher (False Positive $\approx \$50$ cost). Therefore, the business strongly prioritizes **High Recall**.*

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models.train_xgboost import load_model
from src.preprocessing.pipeline import ChurnPreprocessingPipeline
from src.models.evaluator import evaluate_model

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

artifacts = load_model("../models/xgboost_model.pkl" if os.path.exists("../models") else "models/xgboost_model.pkl")
pipeline = ChurnPreprocessingPipeline.load("../models/preprocessor.pkl" if os.path.exists("../models") else "models/preprocessor.pkl")
model = artifacts["calibrated_model"]
print("Calibrated Model Artifacts Loaded Successfully!")

## 📊 1. Load Test Dataset & Compute 7 Evaluation Metrics

In [ ]:
DATA_PATH = Path("../data/raw") if (Path("../data/raw") / "customer_churn.csv").exists() else Path("data/raw")
df_raw = pd.read_csv(DATA_PATH / "customer_churn.csv")

from src.features.feature_builder import build_features
df_feat = build_features(df_raw)
X_full = pipeline.transform(df_feat)
y_full = df_feat["Churn"].values

# Evaluate metrics on dataset
metrics = evaluate_model(model, X_full, y_full, output_dir="../figures" if os.path.exists("../figures") else "figures")
metrics_df = pd.DataFrame([{
    "ROC-AUC": metrics["roc_auc"],
    "PR-AUC": metrics["pr_auc"],
    "Accuracy": metrics["accuracy"],
    "Precision": metrics["precision"],
    "Recall": metrics["recall"],
    "F1-Score": metrics["f1_score"],
    "Brier Score": metrics["brier_score"]
}])
metrics_df

## ⚖️ 2. Financial Asymmetry Cost Calculation
Comparing total monetary loss from **False Negatives (Lost CLV)** vs. **False Positives (Wasted Offer Cost)**.

In [ ]:
fn_cost = metrics["total_fn_cost"]
fp_cost = metrics["total_fp_cost"]

print(f"• False Negative Count (Missed Churners): {metrics['fn_count']:,} -> Lost CLV Penalty: ${fn_cost:,.2f}")
print(f"• False Positive Count (Unnecessary Offers): {metrics['fp_count']:,} -> Wasted Offer Cost: ${fp_cost:,.2f}")
print(f"• Financial Asymmetry Ratio: FN Loss is {fn_cost / fp_cost:.1f}x greater than FP Loss!")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=["False Negative Loss (Lost CLV)", "False Positive Loss (Wasted Offer)"], y=[fn_cost, fp_cost], palette=["#e74c3c", "#f39c12"], ax=ax)
ax.set_title("Asymmetric Financial Cost Breakdown ($)", fontsize=13, fontweight='bold')
ax.set_ylabel("Total Financial Loss ($)")
plt.tight_layout()
plt.show()

## 💡 3. Key Takeaway & Strategic Conclusion

1. **Recall Priority**: Because $\text{Cost}(\text{FN}) \gg \text{Cost}(\text{FP})$, failing to identify a churning customer is catastrophically expensive compared to offering an unnecessary discount voucher.
2. **Calibrated Probabilities**: Sigmoid probability calibration ensures that $P(\text{Churn})$ outputs map cleanly to true empirical churn rates, enabling precise Level 4 Expected Net Gain calculations.

---